# Matching Texte-Image (Dual Encoder + Contrastive Learning)

Ce notebook entraîne un modèle pour associer les textes produits aux images correspondantes.
Voir `PLAN_MATCHING_TEXTE_IMAGE.md` pour les justifications techniques.

In [15]:
# À exécuter UNE FOIS si vous avez "ModuleNotFoundError: No module named 'torch'"
# Ensuite : Kernel → Redémarrer le noyau, puis réexécuter tout le notebook
#
# GPU NVIDIA : la commande ci-dessous suffit (CUDA inclus selon la version).
# GPU AMD Radeon : le pip standard ne prend pas la Radeon. Il faut PyTorch avec ROCm :
#   voir https://rocm.docs.amd.com/projects/radeon-ryzen/en/latest/docs/install/installrad/windows/install-pytorch.html
#   (Windows 11, drivers AMD à jour, puis installation des wheels PyTorch-ROCm.)
%pip install torch torchvision sentence-transformers --quiet

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Tristan\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [16]:
import os
import sys
import importlib
from pathlib import Path

ROOT = Path("..").resolve() if (Path("..") / "data brut").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT))

DATA_BRUT = ROOT / "data brut"
IMAGE_TRAIN_DIR = ROOT / "data" / "processed" / "image_train"

import numpy as np
import pandas as pd
# Recharger le module pour prendre en compte les modifs sans redémarrer le kernel
import src.multimodal.data_loader as _data_loader_module
importlib.reload(_data_loader_module)
from src.multimodal.data_loader import load_text_image_pairs, create_pairs_dataset, load_image
from src.multimodal.encoders import TextEncoder, ImageEncoder
from src.multimodal.matching_model import DualEncoderModel, contrastive_loss
from src.multimodal.utils import recall_at_k, find_matching_images

In [17]:
# Chargement des paires (texte, image) — chemins relatifs à ROOT
pairs_df = load_text_image_pairs(DATA_BRUT, IMAGE_TRAIN_DIR, preprocess_text=True, root=ROOT)
train_pairs, val_pairs = create_pairs_dataset(pairs_df, train_size=0.8, random_state=42)
print(train_pairs.head())

                                                text  \
0  porte bébé violet et rouge trois-en-un mère mu...   
1  jesus - cahiers du libre avenir prêtre autrement.   
2  chambre paillasson en forme de coeur tapis flu...   
3  2pcs en alliage d'aluminium portail du carter ...   
4  harnais chien arnais noir anti traction gilet ...   

                                          image_path   productid     imageid  \
0  data\processed\image_train\image_1187504001_pr...  3050424970  1187504001   
1  data\processed\image_train\image_885888766_pro...   131641431   885888766   
2  data\processed\image_train\image_1313030973_pr...  4197486437  1313030973   
3  data\processed\image_train\image_1265009801_pr...  3929174950  1265009801   
4  data\processed\image_train\image_1313455838_pr...  4183293159  1313455838   

   prdtypecode  
0         1320  
1           10  
2         1560  
3         1280  
4         2220  


In [18]:
# Création des encoders et du modèle (utilise BEST_CONFIG si la phase pilote a été exécutée)
try:
    _cfg = BEST_CONFIG
except NameError:
    _cfg = None

EMBEDDIM = _cfg["embed_dim"] if _cfg else 256
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
n_cpu_threads = min(16, os.cpu_count() or 8)
torch.set_num_threads(n_cpu_threads)
if DEVICE == "cuda":
    print(f"Calculs : GPU — {torch.cuda.get_device_name(0)} | CPU : {n_cpu_threads} threads (préparation des données)")
else:
    print(f"Calculs : CPU uniquement — {n_cpu_threads} threads")
    print("  (Pas de GPU détecté. Carte AMD Radeon ? Il faut PyTorch avec ROCm : voir doc AMD ROCm / install-pytorch Windows.)")

text_encoder = TextEncoder(
    model_name=_cfg["text_model"] if _cfg else "paraphrase-multilingual-MiniLM-L12-v2",
    embedding_dim=EMBEDDIM,
    device=DEVICE,
)
image_encoder = ImageEncoder(embedding_dim=EMBEDDIM, pretrained=True, device=DEVICE)

# Phase 1 : geler le backbone image, entraîner seulement les projections
for p in image_encoder.backbone.parameters():
    p.requires_grad = False

model = DualEncoderModel(text_encoder, image_encoder, temperature=0.07, device=DEVICE)
optimizer = __import__("torch").optim.Adam(model.parameters(), lr=1e-4)

Calculs : CPU uniquement — 16 threads
  (Pas de GPU détecté. Carte AMD Radeon ? Il faut PyTorch avec ROCm : voir doc AMD ROCm / install-pytorch Windows.)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
# Phase 1 : entraînement (projections seules, backbone image gelé) — 10 epochs, batch 64
import torch
import time
from torch.utils.data import Dataset, DataLoader

class TextImagePairsDataset(Dataset):
    """Dataset pour charger (texte, image) — permet chargement parallèle sur plusieurs cœurs CPU."""
    def __init__(self, pairs_df, base_dir):
        self.pairs_df = pairs_df
        self.base_dir = base_dir
    def __len__(self):
        return len(self.pairs_df)
    def __getitem__(self, i):
        row = self.pairs_df.iloc[i]
        img = load_image(row["image_path"], base_dir=self.base_dir)
        return row["text"], torch.from_numpy(img).float()

def _collate_text_image(batch):
    texts = [b[0] for b in batch]
    imgs = torch.stack([b[1] for b in batch])
    return texts, imgs

def eval_recall(model, pairs_df, max_samples=500, batch_size=64):
    """Calcule Recall@K sur un sous-ensemble de paires (pour validation rapide)."""
    from src.multimodal.utils import recall_at_k
    n = min(len(pairs_df), max_samples)
    sub = pairs_df.sample(n=n, random_state=42)
    text_list = sub["text"].tolist()
    productids = sub["productid"].values
    te_list, ie_list = [], []
    for start in range(0, n, batch_size):
        batch = sub.iloc[start:start+batch_size]
        t = batch["text"].tolist()
        imgs = np.stack([load_image(p, base_dir=ROOT) for p in batch["image_path"]])
        te_list.append(model.get_text_embeddings(t, as_tensor=False))
        ie_list.append(model.get_image_embeddings(imgs, as_tensor=False))
    te = np.vstack(te_list)
    ie = np.vstack(ie_list)
    return recall_at_k(te, ie, productids, productids, k_values=[1, 5, 10])

BATCH_SIZE = 128  # Plus de négatives → meilleur contrastive, vise ~80%% Recall@10 (réduire à 64 si OOM)
EPOCHS_PHASE1 = 18
VAL_EVERY = 2
VAL_SAMPLES = 500
N_TRAIN = len(train_pairs)
bar = "━" * 18
save_dir = ROOT / "models"
save_dir.mkdir(parents=True, exist_ok=True)
best_recall10 = 0.0  # Objectif : ~80%% Recall@10
best_state = None

from torch.optim.lr_scheduler import CosineAnnealingLR
scheduler = CosineAnnealingLR(optimizer, T_max=EPOCHS_PHASE1, eta_min=1e-6)

NUM_WORKERS = 0  # 0 = tout dans le processus principal → meilleure utilisation CPU sans GPU
train_dataset = TextImagePairsDataset(train_pairs, ROOT)
dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, collate_fn=_collate_text_image, pin_memory=(DEVICE=="cuda"))

start_time = time.time()
for epoch in range(EPOCHS_PHASE1):
    epoch_start = time.time()
    model.text_encoder.model.eval()
    image_encoder.projection.train()
    if getattr(text_encoder, "_projection", None) is not None:
        text_encoder._projection.train()
    epoch_loss = 0.0
    epoch_acc = 0.0
    n_batches = 0
    for texts, images in dataloader:
        images = images.to(DEVICE).permute(0, 3, 1, 2)
        with torch.no_grad():
            text_emb = model.get_text_embeddings(texts, as_tensor=True)
        image_emb = model.get_image_embeddings(images, as_tensor=True)
        loss = model.compute_loss(text_emb, image_emb)
        with torch.no_grad():
            sim = (text_emb @ image_emb.T)
            B = sim.shape[0]
            batch_acc = (sim.argmax(dim=1) == torch.arange(B, device=sim.device)).float().mean().item()
        epoch_acc += batch_acc
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    scheduler.step()
    epoch_elapsed = time.time() - epoch_start
    avg_loss = epoch_loss / n_batches
    avg_acc = epoch_acc / n_batches
    ms_per_step = epoch_elapsed / n_batches * 1000
    print(f"Epoch {epoch+1}/{EPOCHS_PHASE1} (lr={scheduler.get_last_lr()[0]:.2e})")
    print(f"{n_batches}/{n_batches} {bar} {int(epoch_elapsed)}s {int(ms_per_step)}ms/step - accuracy: {avg_acc:.4f} - loss: {avg_loss:.4f}")
    # Validation tous les VAL_EVERY epochs + sauvegarde du meilleur modèle (Recall@10, objectif ~80%%)
    if (epoch + 1) % VAL_EVERY == 0:
        text_encoder.model.eval()
        image_encoder.eval()
        rec = eval_recall(model, val_pairs, max_samples=VAL_SAMPLES)
        print(f"  → Val Recall@1/5/10: {rec['recall@1']:.3f} / {rec['recall@5']:.3f} / {rec['recall@10']:.3f}")
        if rec["recall@10"] > best_recall10:
            best_recall10 = rec["recall@10"]
            best_state = {
                "image_encoder": {
                    "backbone": {k: v.cpu().clone() for k, v in image_encoder.backbone.state_dict().items()},
                    "projection": {k: v.cpu().clone() for k, v in image_encoder.projection.state_dict().items()},
                },
                "text_projection": None,
            }
            if getattr(text_encoder, "_projection", None) is not None:
                best_state["text_projection"] = {k: v.cpu().clone() for k, v in text_encoder._projection.state_dict().items()}
            torch.save(best_state, save_dir / "matching_best_phase1.pt")
            print(f"  → Meilleur modèle sauvegardé (Recall@10={best_recall10:.3f})")

elapsed_phase1 = time.time() - start_time
print(f"Phase 1 — Temps d'entraînement : {elapsed_phase1:.0f} s")

---
### Phase 2 : fine-tuning du backbone image

Après la Phase 1, on dégèle le ResNet50 et on entraîne 8 epochs avec un LR plus faible (1e-5) pour adapter les features visuelles au catalogue. Objectif : viser ~80 % Recall@10.

In [ ]:
# Phase 2 : dégeler le backbone image, fine-tuner avec un LR plus faible — 8 epochs (vise ~80%% Recall@10)
try:
    _ = best_recall10
except NameError:
    best_recall10 = 0.0

for p in image_encoder.backbone.parameters():
    p.requires_grad = True
optimizer_phase2 = torch.optim.Adam(model.parameters(), lr=1e-5)  # LR plus bas pour ne pas casser les features

EPOCHS_PHASE2 = 8  # Plus d'epochs pour viser ~80%% Recall@10
start_phase2 = time.time()
for epoch in range(EPOCHS_PHASE2):
    epoch_start = time.time()
    model.text_encoder.model.eval()
    image_encoder.train()
    if getattr(text_encoder, "_projection", None) is not None:
        text_encoder._projection.train()
    epoch_loss = 0.0
    epoch_acc = 0.0
    n_batches = 0
    for texts, images in dataloader:
        images = images.to(DEVICE).permute(0, 3, 1, 2)
        with torch.no_grad():
            text_emb = model.get_text_embeddings(texts, as_tensor=True)
        image_emb = model.get_image_embeddings(images, as_tensor=True)
        loss = model.compute_loss(text_emb, image_emb)
        with torch.no_grad():
            sim = (text_emb @ image_emb.T)
            B = sim.shape[0]
            batch_acc = (sim.argmax(dim=1) == torch.arange(B, device=sim.device)).float().mean().item()
        epoch_acc += batch_acc
        optimizer_phase2.zero_grad()
        loss.backward()
        optimizer_phase2.step()
        epoch_loss += loss.item()
        n_batches += 1
    epoch_elapsed = time.time() - epoch_start
    avg_loss = epoch_loss / n_batches
    avg_acc = epoch_acc / n_batches
    ms_per_step = epoch_elapsed / n_batches * 1000
    print(f"Phase 2 — Epoch {epoch+1}/{EPOCHS_PHASE2}")
    print(f"{n_batches}/{n_batches} {bar} {int(epoch_elapsed)}s {int(ms_per_step)}ms/step - accuracy: {avg_acc:.4f} - loss: {avg_loss:.4f}")
    if (epoch + 1) % 2 == 0:
        text_encoder.model.eval()
        image_encoder.eval()
        rec = eval_recall(model, val_pairs, max_samples=VAL_SAMPLES)
        print(f"  → Val Recall@1/5/10: {rec['recall@1']:.3f} / {rec['recall@5']:.3f} / {rec['recall@10']:.3f}")
        if rec["recall@10"] > best_recall10:
            best_recall10 = rec["recall@10"]
            best_state = {
                "image_encoder": {
                    "backbone": {k: v.cpu().clone() for k, v in image_encoder.backbone.state_dict().items()},
                    "projection": {k: v.cpu().clone() for k, v in image_encoder.projection.state_dict().items()},
                },
                "text_projection": None,
            }
            if getattr(text_encoder, "_projection", None) is not None:
                best_state["text_projection"] = {k: v.cpu().clone() for k, v in text_encoder._projection.state_dict().items()}
            torch.save(best_state, save_dir / "matching_best.pt")
            print(f"  → Meilleur modèle sauvegardé (Recall@10={best_recall10:.3f})")

print(f"Phase 2 — Temps : {time.time() - start_phase2:.0f} s")

In [ ]:
# Charger le meilleur modèle sauvegardé (optionnel — pour évaluer le meilleur Recall@5)
best_path = save_dir / "matching_best.pt"
if not best_path.exists():
    best_path = save_dir / "matching_best_phase1.pt"
if best_path.exists():
    state = torch.load(best_path, map_location=DEVICE)
    ie_state = state["image_encoder"]
    image_encoder.backbone.load_state_dict(ie_state["backbone"], strict=True)
    image_encoder.projection.load_state_dict(ie_state["projection"], strict=True)
    if state.get("text_projection") and getattr(text_encoder, "_projection", None) is not None:
        text_encoder._projection.load_state_dict(state["text_projection"], strict=True)
    print(f"Meilleur modèle chargé depuis {best_path.name}")
else:
    print("Aucun checkpoint trouvé, on garde les poids du dernier epoch.")

In [ ]:
# Évaluation : Recall@K sur la validation
text_encoder.model.eval()
image_encoder.eval()

def encode_val_batches(pairs_df, model, batch_size=64, max_samples=2000):
    from src.multimodal.data_loader import load_image
    n = min(len(pairs_df), max_samples)
    pairs_df = pairs_df.sample(n=n, random_state=42)
    texts = pairs_df["text"].tolist()
    productids = pairs_df["productid"].values
    text_emb_list = []
    image_emb_list = []
    for start in range(0, n, batch_size):
        batch = pairs_df.iloc[start:start+batch_size]
        t = batch["text"].tolist()
        imgs = np.stack([load_image(p, base_dir=ROOT) for p in batch["image_path"]])
        text_emb_list.append(model.get_text_embeddings(t, as_tensor=False))
        image_emb_list.append(model.get_image_embeddings(imgs, as_tensor=False))
    return (
        np.vstack(text_emb_list),
        np.vstack(image_emb_list),
        productids,
    )

# Évaluation sur 2000 échantillons de validation (métriques plus stables)
te, ie, ids = encode_val_batches(val_pairs, model, batch_size=64, max_samples=2000)  # 2000 pour métriques plus stables
recall = recall_at_k(te, ie, ids, ids, k_values=[1, 5, 10])
print("Recall@K (validation):", recall)

In [ ]:
# Exemple : recherche d'images par texte
sample_text = train_pairs["text"].iloc[0]
image_paths = train_pairs["image_path"].tolist()[:500]
# Encoder les images par batch pour être plus rapide
from src.multimodal.data_loader import load_image
img_batch = np.stack([load_image(p, base_dir=ROOT) for p in image_paths])
image_embs = model.get_image_embeddings(img_batch)
top = find_matching_images(sample_text, text_encoder, image_embs, image_paths, top_k=3)
print("Texte:", sample_text[:200], "...")
print("Top 3 images:", top)